# AML-GNN — train, test and quantify Multi-PNA on HI-Small (~35 min)

1. **Runtime → Change runtime type → A100 GPU** (or **G4 GPU**), High-RAM. Avoid T4, TPU and CPU.
2. Upload `AML_GNN_colab.zip` (code + data; create it with `python -m aml_gnn bundle train`, it appears in `dist/`) to `MyDrive/AML/`.
3. Check the settings cell, then **Runtime → Run all**.

Steps: prepare graph → 1-minute GPU check → training (time-boxed) → results report → copy everything to `MyDrive/AML/output`.

Total time ≈ `TIME_BUDGET` + ~8 minutes (setup, graph, test, scoring all 5.08M transactions, report).

Afterwards, copy `multipna_model.pt`, `training_metrics.json` and `edge_scores.parquet` into the project's `artifacts/` folder to use the API and dashboard. `results.md` holds the numbers for your report and resume.

In [ ]:
# ==== Settings ====
DRIVE_DIR     = "/content/drive/MyDrive/AML"   # folder holding AML_GNN_colab.zip
TIME_BUDGET   = 25    # minutes of training, then the best epoch is kept (0 = no limit)
TRAIN_BATCHES = 256   # training batches per epoch, as in FraudGT/MEGA (0 = full pass)
SEEDS         = [1]   # first = main run. [1, 2, 3] gives mean ± std (each extra seed adds TIME_BUDGET minutes)

BUNDLE  = f"{DRIVE_DIR}/AML_GNN_colab.zip"
OUT_DIR = f"{DRIVE_DIR}/output"
PROJECT = "/content/AML-GNN"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, torch
assert os.path.exists(BUNDLE), f"Not found: {BUNDLE}"
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
print("GPU:", gpu, "| torch", torch.__version__)
if not any(name in gpu for name in ("A100", "H100", "RTX PRO 6000")):
    print("WARNING: slow GPU for a time-boxed run - Runtime → Change runtime type → A100 or G4")

In [ ]:
# Unzip code + data to fast local disk; install the two graph libraries
# (Colab already has torch, pandas and pyarrow).
!rm -rf {PROJECT} && mkdir -p {PROJECT}
!unzip -q -o "{BUNDLE}" -d {PROJECT}
for p in ("aml_gnn/train.py", "aml_gnn/report.py", "data/raw/HI-Small_Trans.csv", "data/raw/HI-Small_Patterns.txt"):
    assert os.path.exists(f"{PROJECT}/{p}"), f"Bundle is missing {p}"
!pip install -q torch_geometric torchmetrics
!ls -lh {PROJECT}/aml_gnn {PROJECT}/data/raw

In [ ]:
# Step 1 — build the graph (~1-2 min). Check the integrity report:
# 515,088 accounts, 5,078,345 transactions, 5,177 laundering, split 3,248,921 / 965,524 / 863,900.
!cd {PROJECT} && python -m aml_gnn prepare

In [ ]:
# Step 2 — 1-minute GPU check at full settings (3 batches): catches errors before the long run.
# Its scores are meaningless; the run below overwrites its files.
!cd {PROJECT} && python -m aml_gnn train --epochs 1 --max-batches 3 --no-scores 2>&1 | grep --line-buffered -v FutureWarning | grep -E "device|Model parameters|Epoch|TEST|Error|Traceback"

In [ ]:
# Step 3 — training: stops after TIME_BUDGET minutes (or early stopping) and keeps the best epoch,
# then tests it and scores all 5.08M transactions. Extra seeds run first; the main seed runs last.
os.makedirs(f"{PROJECT}/artifacts/seeds", exist_ok=True)
for seed in SEEDS[1:] + SEEDS[:1]:
    extra = "" if seed == SEEDS[0] else "--no-scores"
    print(f"===== seed {seed} =====")
    !cd {PROJECT} && PYTHONUNBUFFERED=1 python -m aml_gnn train --seed {seed} --time-budget {TIME_BUDGET} --train-batches {TRAIN_BATCHES} {extra} 2>&1 | grep --line-buffered -v FutureWarning | tee -a train.log
    shutil.copy(f"{PROJECT}/artifacts/training_metrics.json", f"{PROJECT}/artifacts/seeds/seed_{seed}.json")

In [ ]:
# Step 4 — quantified results: F1 / precision / recall, AUPRC, ROC-AUC, alert budget,
# recall per laundering typology, seed mean ± std, GPU and throughput.
!cd {PROJECT} && python -m aml_gnn report 2>&1 | grep -E "Saved|Error|Traceback"
from IPython.display import Markdown, display
display(Markdown(open(f"{PROJECT}/artifacts/results.md").read()))

In [ ]:
# Step 5 — copy model, scores, predictions, metrics, results and log to Drive
os.makedirs(OUT_DIR, exist_ok=True)
!cp -r {PROJECT}/artifacts/. "{OUT_DIR}/"
!cp {PROJECT}/train.log "{OUT_DIR}/"
!ls -lh "{OUT_DIR}"